# 03 · Encoder–decoder recognition, augmentation and speaker splits

*Voice & Speech Models*, notebook 3 of 4

CTC is one way to turn frames into symbols. The other main family of recognisers **encodes** the audio and then **decodes** the transcript one token at a time, attending to the frames it needs for each token. In the first half of this notebook you build an attention decoder, train it with teacher forcing and compare its behaviour with a CTC model trained on the same data. In the second half you prepare data the way speech systems need it: augmentations that keep the labels valid, and splits by speaker so that an evaluation measures what it claims to measure.

## What you will build

- `teacher_forcing_pairs`: decoder inputs and targets shifted by one step, with an end-of-sequence token and a padding mask;
- `attention_step`: one step of scaled dot-product attention over the encoded frames;
- `add_noise_at_snr`, `time_shift` and `speed_change`: three augmentations, the last one by resampling;
- `group_folds`: cross-validation folds that keep each speaker's recordings together.

## Learning objectives

By the end of this notebook you will be able to:

1. build decoder inputs and outputs for teacher forcing, and explain the difference between training with teacher forcing and generating from the model's own outputs;
2. compute an attention step by hand and on the tiny tensor engine, and read an attention map;
3. contrast how an attention decoder and a CTC model fail when the audio disagrees with what they expect;
4. mix noise at a chosen signal-to-noise ratio, shift audio in time and change its speed by resampling;
5. split data by speaker, and measure how much a random clip-level split overstates accuracy on new speakers and how much augmentation helps.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function in the cell provided and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Cells marked **🧪 Try it** invite you to change one value and run them again.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots, plus the standard-library module `time`.
- The tiny tensor engine from Base Model Training and the pieces from notebooks 01 and 02 (the feature pipeline, the CTC loss and operation, greedy decoding, the edit distance and Adam) are pasted into hidden recap cells.
- Nothing is downloaded: every utterance and every "speaker" is generated from formulas with seeded random generators.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you type against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (with `@`, broadcasting arithmetic, `relu`, `tanh`, `log_softmax`, `softmax`, indexing and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been gradient-checked. Run the cell; open it only if you want to look at the code again.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The two hidden cells below define, under their usual names, the feature pipeline from notebook 01 (`make_tone`, `hann_window`, `frame_signal`, `stft_power`, `hz_to_mel`, `mel_to_hz`, `mel_filterbank`, `log_mel`, `log_mel_spectrogram`) and the recognition pieces from notebook 02 (`ctc_collapse`, `extend_with_blanks`, `ctc_forward`, `ctc_loss_and_grad`, `ctc_op`, `greedy_decode`, `edit_distance` and the `Adam` optimiser). Run them; open them only if you want to look back at the code.

In [ ]:
# 📦 Recap: the pieces from notebook 01, under their usual names.

def make_tone(frequency, duration, sample_rate, amplitude=1.0, phase=0.0):
    """A sampled sine wave of shape (int(round(duration * sample_rate)),)."""
    n_samples = int(round(duration * sample_rate))
    t = np.arange(n_samples) / sample_rate
    return amplitude * np.sin(2 * np.pi * frequency * t + phase)


def hann_window(length):
    """The periodic Hann window: 0.5 - 0.5 cos(2 pi n / length) for n = 0, ..., length - 1."""
    return 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(length) / length)


def frame_signal(x, frame_length, hop):
    """Overlapping frames of x: shape (1 + (len(x) - frame_length) // hop, frame_length)."""
    n_frames = 1 + (len(x) - frame_length) // hop
    return np.array([x[m * hop: m * hop + frame_length] for m in range(n_frames)])


def stft_power(x, frame_length, hop):
    """Power spectrogram |rfft(window * frame)|^2: shape (M, frame_length // 2 + 1)."""
    frames = frame_signal(x, frame_length, hop) * hann_window(frame_length)
    return np.abs(np.fft.rfft(frames, axis=1)) ** 2


def hz_to_mel(frequency):
    """HTK mel scale."""
    return 2595.0 * np.log10(1.0 + np.asarray(frequency, dtype=float) / 700.0)


def mel_to_hz(mel):
    """Inverse of hz_to_mel."""
    return 700.0 * (10.0 ** (np.asarray(mel, dtype=float) / 2595.0) - 1.0)


def mel_filterbank(n_mels, n_fft, sample_rate, f_min=0.0, f_max=None):
    """Triangular mel filters: shape (n_mels, n_fft // 2 + 1)."""
    f_max = sample_rate / 2 if f_max is None else f_max
    edges = mel_to_hz(np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2))
    freqs = np.arange(n_fft // 2 + 1) * sample_rate / n_fft
    left, centre, right = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    rising = (freqs - left) / (centre - left)
    falling = (right - freqs) / (right - centre)
    return np.maximum(0.0, np.minimum(rising, falling))


def log_mel(power, filterbank, eps=1e-10):
    """Log band energies: (M, n_bins) power and (n_mels, n_bins) filters give (M, n_mels)."""
    return np.log(power @ filterbank.T + eps)


def log_mel_spectrogram(x, sample_rate, frame_length=256, hop=128, n_mels=20):
    """Waveform (N,) -> log-mel features (M, n_mels)."""
    return log_mel(stft_power(x, frame_length, hop), mel_filterbank(n_mels, frame_length, sample_rate))

In [ ]:
# 📦 Recap: the pieces from notebook 02, under their usual names.

def ctc_collapse(path, blank=0):
    """Merge repeated labels, then drop blanks: [1, 1, 0, 1, 2] -> [1, 1, 2]."""
    out, previous = [], None
    for label in path:
        if label != previous and label != blank:
            out.append(int(label))
        previous = label
    return out


def extend_with_blanks(target, blank=0):
    """[a, b] -> [_, a, _, b, _]: the extended target of length 2 * len(target) + 1."""
    extended = np.full(2 * len(target) + 1, blank, dtype=int)
    extended[1::2] = target
    return extended


def ctc_forward(log_probs, target, blank=0):
    """Log forward variables log alpha, shape (T, 2 * len(target) + 1)."""
    ext = extend_with_blanks(target, blank)
    T, S = log_probs.shape[0], len(ext)
    log_alpha = np.full((T, S), -np.inf)
    log_alpha[0, 0] = log_probs[0, ext[0]]
    if S > 1:
        log_alpha[0, 1] = log_probs[0, ext[1]]
    can_skip = np.zeros(S, dtype=bool)
    can_skip[2:] = (ext[2:] != blank) & (ext[2:] != ext[:-2])
    for t in range(1, T):
        stay = log_alpha[t - 1]
        step = np.full(S, -np.inf)
        step[1:] = log_alpha[t - 1, :-1]
        skip = np.full(S, -np.inf)
        skip[2:] = log_alpha[t - 1, :-2]
        skip[~can_skip] = -np.inf
        log_alpha[t] = np.logaddexp(np.logaddexp(stay, step), skip) + log_probs[t, ext]
    return log_alpha


def ctc_loss_and_grad(log_probs, target, blank=0):
    """(-log P(target), d loss / d log_probs of shape (T, C)); (inf, zeros) if the target cannot fit."""
    target = np.asarray(target, dtype=int)
    ext = extend_with_blanks(target, blank)
    log_alpha = ctc_forward(log_probs, target, blank)
    log_beta_hat = ctc_forward(log_probs[::-1], target[::-1], blank)[::-1, ::-1]
    log_p = np.logaddexp.reduce(log_alpha[-1, -2:])
    grad = np.zeros_like(log_probs)
    if not np.isfinite(log_p):
        return float("inf"), grad
    occupancy = log_alpha + log_beta_hat - log_probs[:, ext]
    for k in np.unique(ext):
        grad[:, k] = -np.exp(np.logaddexp.reduce(occupancy[:, ext == k], axis=1) - log_p)
    return float(-log_p), grad


def ctc_op(log_probs, target, blank=0):
    """CTC loss as an operation of the tiny tensor engine."""
    loss, grad = ctc_loss_and_grad(log_probs.data, target, blank)
    return Tensor(loss, requires_grad=log_probs.requires_grad, parents=(log_probs,),
                  backward=lambda g: (g * grad,), op="ctc")


def greedy_decode(log_probs, blank=0):
    """Best-path decoding: the most likely label in every frame, then the collapse rule."""
    return ctc_collapse(np.argmax(log_probs, axis=1), blank)


def edit_distance(reference, hypothesis):
    """Levenshtein distance: the fewest substitutions, deletions and insertions."""
    previous = list(range(len(hypothesis) + 1))
    for i, ref_symbol in enumerate(reference, start=1):
        current = [i]
        for j, hyp_symbol in enumerate(hypothesis, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ref_symbol != hyp_symbol)))
        previous = current
    return previous[-1]


class Adam:
    """Adam on a list of engine parameters (as in Base Model Training)."""

    def __init__(self, params, learning_rate=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.learning_rate, self.beta1, self.beta2, self.eps = params, learning_rate, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in params]
        self.v = [np.zeros_like(p.data) for p in params]
        self.t = 0

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            m[:] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[:] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.eps)

# Part A · Encoder–decoder recognition

## 1 · Data: tone sequences with a grammar

The utterances are again sequences of the tones **a**, **b** and **c** (500, 850 and 1400 Hz), with two differences from notebook 02:

- **A steady rhythm.** Symbol $i$ starts about $160\,i$ ms after a short lead-in, with up to 16 ms of jitter, and every utterance is padded with silence to the same length: 704 ms, or 43 frames. A steady rate lets a tiny decoder find symbol $i$ partly by its position, much as a speaker's syllable rate helps a listener.
- **A grammar.** The first symbol is random, but after that **a** is always followed by **b**, **b** by **c**, and **c** by **a**; utterances hold two to four symbols. So "abca" and "bc" occur, but "aa" and "ac" never do. Language has far softer regularities of the same kind, and a decoder that sees its previous outputs learns them.

The decoder's vocabulary has five tokens: `<s>` (start, index 0), **a**, **b** and **c** (indices 1 to 3) and `</s>` (end, index 4). The CTC model trained later on the same audio uses index 0 as its blank instead; the two models never share outputs.

In [ ]:
SAMPLE_RATE, FRAME_LENGTH, HOP, N_MELS = 8000, 256, 128, 20
TONE_HZ = {1: 500.0, 2: 850.0, 3: 1400.0}
NEXT_SYMBOL = {1: 2, 2: 3, 3: 1}                   # the grammar: a -> b -> c -> a
LEAD, SLOT, TONE, MAX_SYMBOLS = 256, 1280, 640, 4  # in samples: 32 ms lead-in, 160 ms per symbol, 80 ms tones
N_SAMPLES = LEAD + MAX_SYMBOLS * SLOT + 256        # 5,632 samples = 704 ms = 43 frames
BOS, EOS = 0, 4
TOKENS = ["<s>", "a", "b", "c", "</s>"]


def sample_sentence(rng):
    """Two to four symbols that follow the grammar."""
    length = int(rng.integers(2, MAX_SYMBOLS + 1))
    sentence = [int(rng.integers(1, 4))]
    while len(sentence) < length:
        sentence.append(NEXT_SYMBOL[sentence[-1]])
    return sentence


def render(sentence, rng, noise=0.05, drop=None):
    """A fixed-length waveform with symbol i in slot i; the symbol at index `drop` is left silent."""
    x = np.zeros(N_SAMPLES)
    for i, symbol in enumerate(sentence):
        if i == drop:
            continue
        start = LEAD + i * SLOT + int(rng.integers(-128, 129))
        n = TONE + int(rng.integers(-64, 65))
        fade = np.minimum(1.0, np.minimum(np.arange(n), n - 1 - np.arange(n)) / 40)
        x[start:start + n] += fade * make_tone(TONE_HZ[symbol] * rng.uniform(0.97, 1.03), n / SAMPLE_RATE, SAMPLE_RATE,
                                               rng.uniform(0.3, 0.9), rng.uniform(0.0, 2 * np.pi))
    return x + noise * rng.normal(size=N_SAMPLES)


def make_sentences(n, seed, **render_options):
    rng = np.random.default_rng(seed)
    sentences = [sample_sentence(rng) for _ in range(n)]
    features = np.stack([log_mel_spectrogram(render(s, rng, **render_options), SAMPLE_RATE) for s in sentences])
    return features, sentences


def to_text(tokens):
    return "".join("_abc"[t] if 0 < t < 4 else "" for t in tokens)


train_x, train_sentences = make_sentences(400, seed=1)
test_x, test_sentences = make_sentences(100, seed=3)
print("features:", train_x.shape, "(utterances, frames, mel bands)")
print("training sentences:", [to_text(s) for s in train_sentences[:8]])

In [ ]:
feature_mean = train_x.reshape(-1, N_MELS).mean(axis=0)     # training frames only
feature_std = train_x.reshape(-1, N_MELS).std(axis=0) + 1e-8


def prepare(features, context=1):
    """Normalise a batch (B, T, 20) with the training statistics and append `context` frames on each side."""
    z = (features - feature_mean) / feature_std
    padded = np.pad(z, ((0, 0), (context, context), (0, 0)), mode="edge")
    T = z.shape[1]
    return np.concatenate([padded[:, i:i + T] for i in range(2 * context + 1)], axis=2)


train_inputs, test_inputs = prepare(train_x), prepare(test_x)
print("prepared:", train_inputs.shape)

## 2 · Teacher forcing: inputs and outputs shifted by one

An autoregressive decoder predicts token $i$ from the audio and the tokens before it: $P(z \mid x) = \prod_i P(z_i \mid z_{<i}, x)$. During training we know the whole transcript, so we feed the **correct** previous tokens and ask for the next one at every step at once. This is **teacher forcing**. For the transcript "abc":

| step | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| decoder input | `<s>` | a | b | c |
| target output | a | b | c | `</s>` |

The decoder must also learn when to stop, so the targets end with `</s>`. Sequences in a batch have different lengths, so we pad them to the same number of steps and give each step a mask: 1.0 for a real step, 0.0 for padding, which the loss then ignores.

### ✍️ Your turn: `teacher_forcing_pairs`

Write `teacher_forcing_pairs(target, max_steps, bos=0, eos=4)` that returns a tuple of three lists, each of length `max_steps`:

- `inputs`: `[bos] + target`, then padded with `eos`;
- `outputs`: `target + [eos]`, then padded with `eos`;
- `mask`: `1.0` for each of the `len(target) + 1` real steps, then `0.0` for the padding.

You may assume `len(target) + 1 <= max_steps`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def teacher_forcing_pairs_reference(target, max_steps, bos=0, eos=4):
    target = [int(t) for t in target]
    padding = max_steps - len(target) - 1
    inputs = [bos] + target + [eos] * padding
    outputs = target + [eos] + [eos] * padding
    mask = [1.0] * (len(target) + 1) + [0.0] * padding
    return inputs, outputs, mask

In [ ]:
def teacher_forcing_pairs(target, max_steps, bos=0, eos=4):
    """Decoder inputs, target outputs and loss mask, each padded to max_steps."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
teacher_forcing_pairs = practice("teacher_forcing_pairs", teacher_forcing_pairs, teacher_forcing_pairs_reference, cases=[
    ([1, 2, 3], 5),
    ([2, 3], 5),
    ([3, 1, 2, 3], 5),    # the longest sentence: no padding at all
    ([], 2),              # an empty transcript: predict </s> straight away
])
print(teacher_forcing_pairs([1, 2, 3], 5))

**Expected behaviour.** For "abc" with five steps: inputs `[0, 1, 2, 3, 4]`, outputs `[1, 2, 3, 4, 4]` and mask `[1.0, 1.0, 1.0, 1.0, 0.0]`. The last input is padding and its output is masked out.

## 3 · One attention step

The encoder turns the $T$ frames into vectors $h_1, \dots, h_T$ of size $d$; here a dense layer with a `tanh`, plus a learned position vector for each frame. At decoder step $i$, a **query** $q_i$ (built from the previous token and the step number) scores every frame, the scores become weights with a softmax, and the weighted average of the frames is the **context** used to predict the next token:

$$e_{t} = \frac{q \cdot k_t}{\sqrt{d}}, \qquad a_t = \frac{e^{e_t}}{\sum_{u} e^{e_u}}, \qquad c = \sum_{t} a_t\, v_t .$$

Here the keys $k_t$ and values $v_t$ are both the encoder vectors $h_t$. Shapes for one step: the query is `(d,)`, the keys `(T, d)`, the values `(T, d_v)`, the weights `(T,)` and the context `(d_v,)`. Dividing by $\sqrt d$ keeps the scores from growing with the vector size and saturating the softmax.

Unlike CTC, which makes one decision per frame, the decoder makes one decision per **token**, can look at any frame for any token, and decides the output length itself by emitting `</s>`.

In [ ]:
keys_demo = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # three frames, d = 2
query_demo = np.array([2.0, 0.0])
scores_demo = keys_demo @ query_demo / np.sqrt(2)
print("scores:", np.round(scores_demo, 3), "-> the first and third frames match the query best")

### ✍️ Your turn: `attention_step`

Write `attention_step(query, keys, values)` that returns a tuple `(context, weights)`, following the formulas above, in NumPy.

Hints: `keys @ query` gives all $T$ dot products at once; subtract the largest score before `np.exp` so that large scores cannot overflow (this does not change the softmax).

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def attention_step_reference(query, keys, values):
    scores = keys @ query / np.sqrt(len(query))
    weights = np.exp(scores - scores.max())
    weights = weights / weights.sum()
    return weights @ values, weights

In [ ]:
def attention_step(query, keys, values):
    """Scaled dot-product attention for one query: (context (d_v,), weights (T,))."""
    # ✍️ Your code here (scores, a stable softmax, then the weighted average)
    raise NotImplementedError

In [ ]:
attention_rng = np.random.default_rng(1)
attention_step = practice("attention_step", attention_step, attention_step_reference, cases=[
    (query_demo, keys_demo, keys_demo),
    (attention_rng.normal(size=4), attention_rng.normal(size=(6, 4)), attention_rng.normal(size=(6, 3))),
    (np.array([100.0, 0.0]), keys_demo, keys_demo),   # huge scores: the softmax must not overflow
])
context_demo, weights_demo = attention_step(query_demo, keys_demo, keys_demo)
print("weights:", np.round(weights_demo, 3), " context:", np.round(context_demo, 3))

**Expected behaviour.** For the demo, the weights are about `[0.45, 0.11, 0.45]`: the first and third frames match the query equally well, so the context is about `[0.89, 0.55]`. With a huge query, nearly all the weight lands on those two frames and nothing overflows.

## 4 · The decoder on the tiny tensor engine, trained with teacher forcing

The engine version below does exactly what your `attention_step` does, for every decoder step of every utterance in a batch at once: `Q @ H.transpose(0, 2, 1)` holds all the scores, a `softmax` over the frames gives the weights, and `A @ H` gives all the contexts. Each step then combines its context with its query and predicts one of the five tokens. The loss is the engine's `cross_entropy` with your mask, so padding steps cost nothing.

In [ ]:
D, V = 32, 5
model_rng = np.random.default_rng(0)
enc_W, enc_b = parameter(60, D, rng=model_rng), zeros_parameter(D)
frame_pos = parameter(43, D, rng=model_rng, scale=0.5)         # one learned vector per frame position
token_emb = parameter(V, D, rng=model_rng, scale=0.5)          # one learned vector per token
step_pos = parameter(MAX_SYMBOLS + 1, D, rng=model_rng, scale=0.5)
W_c, W_q = parameter(D, D, rng=model_rng), parameter(D, D, rng=model_rng)
W_o, b_o = parameter(D, V, rng=model_rng, scale=0.1), zeros_parameter(V)
decoder_params = [enc_W, enc_b, frame_pos, token_emb, step_pos, W_c, W_q, W_o, b_o]


def encode(inputs):
    """(B, T, 60) prepared frames -> (B, T, D) encoder vectors."""
    return (Tensor(inputs) @ enc_W + enc_b).tanh() + frame_pos


def decoder_logits(H, tokens):
    """Teacher-forced decoding: (B, T, D) encoder vectors and (B, L) input tokens -> (B, L, V) logits and (B, L, T) weights."""
    Q = token_emb[tokens] + step_pos[np.arange(tokens.shape[1])]       # one query per step
    A = (Q @ H.transpose(0, 2, 1) / np.sqrt(D)).softmax(axis=-1)       # attention weights over the frames
    C = A @ H                                                          # one context per step
    return (C @ W_c + Q @ W_q).tanh() @ W_o + b_o, A


pairs = [teacher_forcing_pairs(s, MAX_SYMBOLS + 1) for s in train_sentences]
dec_inputs = np.array([p[0] for p in pairs])
dec_outputs = np.array([p[1] for p in pairs])
dec_mask = np.array([p[2] for p in pairs])

optimizer = Adam(decoder_params, learning_rate=0.01)
batch_rng = np.random.default_rng(7)
decoder_losses = []
start = time.perf_counter()
for step in range(300):
    batch = batch_rng.choice(len(train_inputs), size=32, replace=False)
    logits, _ = decoder_logits(encode(train_inputs[batch]), dec_inputs[batch])
    loss = cross_entropy(logits, dec_outputs[batch], dec_mask[batch])
    zero_grad(decoder_params)
    loss.backward()
    optimizer.step()
    decoder_losses.append(float(loss.data))
print(f"teacher-forced loss {decoder_losses[0]:.3f} -> {decoder_losses[-1]:.5f} in {time.perf_counter() - start:.1f} s")

## 5 · Generating without a teacher

At test time there is no transcript to feed in. The decoder starts from `<s>`, predicts a token, appends its **own** prediction to the input and repeats until it emits `</s>` or reaches the step limit. Training never showed it its own mistakes, so an early error can lead it somewhere it has never been; this mismatch between training and use is called **exposure bias**.

In [ ]:
def generate(inputs):
    """Greedy autoregressive decoding for a batch: returns the token lists and the attention weights."""
    H = encode(inputs)
    tokens = np.full((len(inputs), 1), BOS)
    for _ in range(MAX_SYMBOLS + 1):
        logits, weights = decoder_logits(H, tokens)
        tokens = np.concatenate([tokens, np.argmax(logits.data[:, -1], axis=-1)[:, None]], axis=1)
    sentences = []
    for row in tokens[:, 1:]:
        row = list(row)
        sentences.append([int(t) for t in row[:row.index(EOS)]] if EOS in row else [int(t) for t in row])
    return sentences, weights.data


def symbol_error_rate(references, hypotheses):
    return sum(edit_distance(r, h) for r, h in zip(references, hypotheses)) / sum(len(r) for r in references)


decoded, attention = generate(test_inputs)
decoder_ser = symbol_error_rate(test_sentences, decoded)
print(f"attention decoder, held-out symbol error rate: {decoder_ser:.3f}")
for reference, hypothesis in list(zip(test_sentences, decoded))[:5]:
    print(f"  reference {to_text(reference):5s} -> decoded {to_text(hypothesis)}")

example = 0
steps = len(test_sentences[example]) + 1
plt.figure(figsize=(8, 2.6))
plt.imshow(attention[example, :steps], aspect="auto", cmap="Greys")
for i in range(MAX_SYMBOLS + 1):
    plt.axvline((LEAD + i * SLOT) / HOP, color="C1", lw=0.6)
plt.yticks(range(steps), [f"step {i}" for i in range(steps)])
plt.xlabel("frame (orange lines: the start of each symbol's slot)")
plt.title(f"Attention for '{to_text(test_sentences[example])}': each step looks at its own slot")
plt.tight_layout(); plt.show()

## 6 · The same audio, a CTC model, and a missing symbol

To compare the two approaches fairly, train a CTC model on exactly the same utterances, using your CTC operation from notebook 02. Then test both models on audio that contradicts the grammar: sentences of three or four symbols whose **second tone has been removed**, leaving silence in its slot. What should a recogniser write when it hears "a, silence, c"?

In [ ]:
ctc_rng = np.random.default_rng(0)
ctc_inputs, ctc_test_inputs = prepare(train_x, context=2), prepare(test_x, context=2)
cW1, cb1 = parameter(100, 32, rng=ctc_rng), zeros_parameter(32)
cW2, cb2 = parameter(32, 4, rng=ctc_rng, scale=0.1), zeros_parameter(4)
cb2.data[0] = 2.0                                  # start with the blank, as in notebook 02
ctc_params = [cW1, cb1, cW2, cb2]


def ctc_log_probs(inputs):
    return ((Tensor(inputs) @ cW1 + cb1).relu() @ cW2 + cb2).log_softmax(axis=-1)


ctc_optimizer = Adam(ctc_params, learning_rate=0.01)
ctc_batches = np.random.default_rng(9)
start = time.perf_counter()
for step in range(200):
    batch = ctc_batches.choice(len(ctc_inputs), size=16, replace=False)
    log_probs = ctc_log_probs(ctc_inputs[batch])
    loss = sum(ctc_op(log_probs[j], train_sentences[i]) for j, i in enumerate(batch)) / len(batch)
    zero_grad(ctc_params)
    loss.backward()
    ctc_optimizer.step()
ctc_decoded = [greedy_decode(lp) for lp in ctc_log_probs(ctc_test_inputs).data]
ctc_ser = symbol_error_rate(test_sentences, ctc_decoded)
print(f"CTC model trained in {time.perf_counter() - start:.1f} s, held-out symbol error rate: {ctc_ser:.3f}")

In [ ]:
drop_rng = np.random.default_rng(11)
long_sentences = []
while len(long_sentences) < 40:
    sentence = sample_sentence(drop_rng)
    if len(sentence) >= 3:
        long_sentences.append(sentence)
dropped_x = np.stack([log_mel_spectrogram(render(s, drop_rng, drop=1), SAMPLE_RATE) for s in long_sentences])
heard = [s[:1] + s[2:] for s in long_sentences]                    # what is actually in the audio

decoder_out, _ = generate(prepare(dropped_x))
ctc_out = [greedy_decode(lp) for lp in ctc_log_probs(prepare(dropped_x, context=2)).data]
summary = {
    "attention decoder": {"wrote what was heard": sum(o == h for o, h in zip(decoder_out, heard)),
                          "restored the missing symbol": sum(o == s for o, s in zip(decoder_out, long_sentences)),
                          "stopped early": sum(len(o) < len(h) for o, h in zip(decoder_out, heard))},
    "CTC": {"wrote what was heard": sum(o == h for o, h in zip(ctc_out, heard)),
            "restored the missing symbol": sum(o == s for o, s in zip(ctc_out, long_sentences)),
            "stopped early": sum(len(o) < len(h) for o, h in zip(ctc_out, heard))},
}
for model, counts in summary.items():
    print(f"{model:17s}: " + ", ".join(f"{name} {count}/40" for name, count in counts.items()))
for sentence, h, d, c in list(zip(long_sentences, heard, decoder_out, ctc_out))[:5]:
    print(f"  spoken {to_text(sentence):5s} heard {to_text(h):4s} -> decoder {to_text(d):5s} CTC {to_text(c)}")

**What to notice.** CTC writes what it hears, frame by frame: each output depends only on the audio around it, so the missing symbol is simply missing. The attention decoder conditions on its own previous outputs and has learned the grammar, so when the audio contradicts the grammar it either **restores** the symbol it expected (a plausible output that is not in the audio) or **stops early**, dropping symbols it did hear. Neither behaviour is always wrong: restoring a symbol masked by a cough is helpful; inventing one the speaker never said is a hallucination. This is why encoder–decoder recognisers are tested on silence, noise and unusual sentences, and why their outputs need checking at chunk boundaries.

### 🧪 Try it

Change `drop=1` to `drop=0` in the cell above (remove the *first* symbol) and run it again. The decoder's first query has no previous symbol to lean on, so how do its failures change?

# Part B · Speech data and adaptation

## 7 · Speakers, recordings and what a split claims

Real speech comes from **speakers**, each with their own pitch, voice quality, speaking rate and microphone. Clips from one speaker resemble each other far more than clips from different speakers. If clips are split at random, the test set contains speakers the model has already heard, and the score answers the question "how well does it recognise *these* speakers?". To claim anything about **new** speakers, every speaker's clips must go to exactly one side of the split.

The generator below creates 16 synthetic speakers. Each has a base pitch between 220 and 520 Hz, a voice quality (the strength of the second and third harmonics), a speaking rate and a background noise level. Each speaker says four "words", which are pitch contours: **rise**, **fall**, **flat** and **peak** (up, then down), five times each. The word is defined by the *shape* of the contour, relative to the speaker's own pitch, just as a word does not change when a deeper voice says it.

In [ ]:
CLIP = 4000                                # 0.5 s at 8 kHz
WORDS = ["rise", "fall", "flat", "peak"]


def contour(word, n):
    """Pitch multiplier over n samples: 1 to 1.74 (rise), the reverse (fall), constant 1.32 (flat) or up and down (peak)."""
    u = np.linspace(0.0, 1.0, n)
    shape = {"rise": u, "fall": 1 - u, "flat": np.full(n, 0.5), "peak": 1 - np.abs(2 * u - 1)}[word]
    return 2.0 ** (0.8 * shape)


def make_speaker(rng):
    return {"f0": float(np.exp(rng.uniform(np.log(220), np.log(520)))), "h2": float(rng.uniform(0, 0.6)),
            "h3": float(rng.uniform(0, 0.4)), "rate": float(rng.uniform(0.85, 1.15)), "noise": float(rng.uniform(0.01, 0.05))}


def say(word, speaker, rng):
    """One 0.5 s clip of `word` in this speaker's voice, at a random position in the clip."""
    n = int(2400 * speaker["rate"] * rng.uniform(0.9, 1.1))
    f0 = speaker["f0"] * rng.uniform(0.97, 1.03) * contour(word, n)
    phase = 2 * np.pi * np.cumsum(f0) / SAMPLE_RATE
    fade = np.minimum(1.0, np.minimum(np.arange(n), n - 1 - np.arange(n)) / 80)
    voice = fade * (np.sin(phase) + speaker["h2"] * np.sin(2 * phase) + speaker["h3"] * np.sin(3 * phase))
    clip = np.zeros(CLIP)
    start = int(rng.integers(200, CLIP - n - 200))
    clip[start:start + n] = rng.uniform(0.3, 0.8) * voice
    return clip + speaker["noise"] * rng.normal(size=CLIP)


corpus_rng = np.random.default_rng(0)
speakers = [make_speaker(corpus_rng) for _ in range(16)]
clips, words, speaker_ids = [], [], []
for speaker_id, speaker in enumerate(speakers):
    for word_id, word in enumerate(WORDS):
        for _ in range(5):
            clips.append(say(word, speaker, corpus_rng)); words.append(word_id); speaker_ids.append(speaker_id)
clips, words, speaker_ids = np.array(clips), np.array(words), np.array(speaker_ids)
print(f"{len(clips)} clips from {len(speakers)} speakers; base pitches from "
      f"{min(s['f0'] for s in speakers):.0f} to {max(s['f0'] for s in speakers):.0f} Hz")

## 8 · Augmentations that keep the label true

**Augmentation** creates new training examples by changing a recording in ways that do not change its transcript. Three classic ones for speech:

- **Noise at a chosen signal-to-noise ratio.** The SNR in decibels compares average powers, $\text{SNR} = 10 \log_{10}(P_\text{signal} / P_\text{noise})$, with $P = \frac{1}{N}\sum_n x[n]^2$. To mix a noise recording at a target SNR, scale it by $\sqrt{P_\text{signal} / (P_\text{noise} \cdot 10^{\text{SNR}/10})}$. At 0 dB the noise is as powerful as the speech; at 20 dB it has a hundredth of the power.
- **A time shift.** Move the audio later (or earlier) and fill the gap with silence, so the model cannot rely on where in the clip a word starts.
- **A speed change.** Play the recording faster or slower by resampling it: read the waveform at positions $0, r, 2r, \dots$ with linear interpolation and play the result at the original rate. A factor $r = 1.1$ makes the clip 10% shorter **and** 10% higher in pitch, which imitates a faster, higher voice; this is called speed perturbation in speech recognition.

Augment only the training data, after splitting, and keep the test conditions fixed and documented.

In [ ]:
def power(x):
    return float(np.mean(x ** 2))


demo_voice = say("rise", speakers[0], np.random.default_rng(1))
print(f"signal power {power(demo_voice):.4f}; white noise with standard deviation 1 has power about 1")

### ✍️ Your turn: `add_noise_at_snr`

Write `add_noise_at_snr(signal, noise, snr_db)` that returns `signal + scale * noise` with the scale from the formula above. `noise` has the same shape as `signal`. Do not modify either input.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def add_noise_at_snr_reference(signal, noise, snr_db):
    scale = np.sqrt(np.mean(signal ** 2) / (np.mean(noise ** 2) * 10 ** (snr_db / 10)))
    return signal + scale * noise

In [ ]:
def add_noise_at_snr(signal, noise, snr_db):
    """Mix `noise` into `signal` so that the signal-to-noise ratio is snr_db decibels."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
snr_rng = np.random.default_rng(2)
add_noise_at_snr = practice("add_noise_at_snr", add_noise_at_snr, add_noise_at_snr_reference, cases=[
    (np.sin(np.arange(50) / 3), snr_rng.normal(size=50), 10.0),
    (np.ones(4), np.array([1.0, -1.0, 1.0, -1.0]), 0.0),     # equal power at 0 dB
    (demo_voice, snr_rng.normal(size=CLIP), 20.0),
])
noisy = add_noise_at_snr(demo_voice, snr_rng.normal(size=CLIP), 5.0)
print(f"measured SNR: {10 * np.log10(power(demo_voice) / power(noisy - demo_voice)):.2f} dB (asked for 5 dB)")

### ✍️ Your turn: `time_shift`

Write `time_shift(signal, shift)` that returns an array of the same length as `signal`: with `shift > 0`, the audio starts `shift` samples later and the first `shift` samples are zeros; with `shift < 0`, it starts earlier and the last `-shift` samples are zeros. You may assume `abs(shift) < len(signal)`.

Hint: start from `out = np.zeros_like(signal)` and copy one slice into another.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def time_shift_reference(signal, shift):
    out = np.zeros_like(signal)
    if shift >= 0:
        out[shift:] = signal[:len(signal) - shift]
    else:
        out[:shift] = signal[-shift:]
    return out

In [ ]:
def time_shift(signal, shift):
    """Delay (shift > 0) or advance (shift < 0) the audio, filling the gap with zeros."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
ramp = np.arange(1.0, 7.0)
time_shift = practice("time_shift", time_shift, time_shift_reference, cases=[
    (ramp, 2), (ramp, -2), (ramp, 0), (ramp, 5),
])
print(time_shift(ramp, 2), time_shift(ramp, -2))

### ✍️ Your turn: `speed_change`

Write `speed_change(signal, factor)` that resamples by linear interpolation:

- the output has `n_out = int(np.floor((len(signal) - 1) / factor)) + 1` samples;
- sample `j` of the output is the signal read at position `j * factor`, interpolated linearly between neighbouring samples: `np.interp(positions, np.arange(len(signal)), signal)`.

A factor above 1 shortens the clip and raises its pitch; below 1, the opposite. (Linear interpolation is a simple resampler: it is fine for these low tones, but a production resampler filters first, as you saw in notebook 01.)

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def speed_change_reference(signal, factor):
    n_out = int(np.floor((len(signal) - 1) / factor)) + 1
    positions = np.arange(n_out) * factor
    return np.interp(positions, np.arange(len(signal)), signal)

In [ ]:
def speed_change(signal, factor):
    """Play `signal` `factor` times faster by reading it at positions 0, factor, 2 * factor, ..."""
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
speed_change = practice("speed_change", speed_change, speed_change_reference, cases=[
    (np.arange(5.0), 2.0),        # [0, 2, 4]
    (np.arange(5.0), 0.5),        # nine samples, halfway values in between
    (np.arange(5.0), 1.0),        # unchanged
    (demo_voice, 1.25),
])


def peak_hz(x):
    return float(np.argmax(np.abs(np.fft.rfft(x))) * SAMPLE_RATE / len(x))


steady = make_tone(400.0, 0.5, SAMPLE_RATE)
for factor in (0.8, 1.0, 1.25):
    faster = speed_change(steady, factor)
    print(f"factor {factor:4.2f}: {len(faster) / SAMPLE_RATE:.3f} s long, peak at {peak_hz(faster):.0f} Hz")

**Expected behaviour.** A 400 Hz tone at factor 1.25 lasts 0.4 s and peaks at 500 Hz; at factor 0.8 it lasts 0.625 s and peaks at 320 Hz. Duration and pitch move together, which is exactly what makes speed perturbation imitate other voices. The plot shows one clip with each augmentation.

In [ ]:
def fit_length(x, n):
    """Crop or zero-pad x to exactly n samples."""
    return x[:n] if len(x) >= n else np.pad(x, (0, n - len(x)))


variants = {"original": demo_voice,
            "noise at 5 dB": add_noise_at_snr(demo_voice, np.random.default_rng(3).normal(size=CLIP), 5.0),
            "shifted by 100 ms": time_shift(demo_voice, 800),
            "speed x 1.25": fit_length(speed_change(demo_voice, 1.25), CLIP)}
fig, axes = plt.subplots(1, 4, figsize=(11, 2.6), sharey=True)
for axis, (name, x) in zip(axes, variants.items()):
    axis.imshow(log_mel_spectrogram(x, SAMPLE_RATE).T, origin="lower", aspect="auto", cmap="magma")
    axis.set_title(name, fontsize=9); axis.set_xlabel("frame")
axes[0].set_ylabel("mel band")
plt.tight_layout(); plt.show()

## 9 · Splits that keep each speaker together

With 16 speakers we can afford **cross-validation by speaker**: divide the speakers into $K$ groups, train on $K - 1$ of them, test on the speakers held out, and repeat so that every speaker is tested once. Averaging over the folds makes the estimate far less dependent on which few speakers happened to land in the test set.

### ✍️ Your turn: `group_folds`

Write `group_folds(groups, n_folds)`. `groups` is an integer array with one speaker identifier per clip. Return a list of `n_folds` arrays; array `f` holds the **sorted** indices of every clip whose speaker is in `np.unique(groups)[f::n_folds]`. Speakers are dealt to folds in turn, so the result is deterministic.

Hint: `np.isin(groups, speakers_in_fold)` marks the clips; `np.flatnonzero` turns the marks into sorted indices.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def group_folds_reference(groups, n_folds):
    unique = np.unique(groups)
    return [np.flatnonzero(np.isin(groups, unique[f::n_folds])) for f in range(n_folds)]

In [ ]:
def group_folds(groups, n_folds):
    """Test-index arrays for n_folds folds, with every group's clips in exactly one fold."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
group_folds = practice("group_folds", group_folds, group_folds_reference, cases=[
    (np.array([0, 0, 1, 1, 2, 2, 3, 3]), 2),
    (np.array([5, 3, 5, 9, 3, 9, 1]), 3),      # identifiers need not be 0, 1, 2, ...
    (speaker_ids, 4),
])
folds = group_folds(speaker_ids, 4)
for f, test_index in enumerate(folds):
    print(f"fold {f}: test speakers {sorted(set(speaker_ids[test_index].tolist()))}, {len(test_index)} clips")

## 10 · Put it together

### A · Two recognisers, one test

The first set of checks restates Part A: both models transcribe held-out sentences, and on audio that breaks the grammar, CTC reports what it heard while the attention decoder does not.

In [ ]:
assert decoder_losses[-1] < 0.05 * decoder_losses[0], "teacher forcing should drive the training loss down"
assert decoder_ser <= 0.05, f"the attention decoder should transcribe held-out sentences, got SER {decoder_ser:.3f}"
assert ctc_ser <= 0.05, f"the CTC model should transcribe held-out sentences, got SER {ctc_ser:.3f}"
assert summary["CTC"]["wrote what was heard"] >= 36, "CTC should write what is in the audio"
assert summary["attention decoder"]["wrote what was heard"] <= 10, "the decoder should lean on the grammar"
print(f"✅ Held-out symbol error rate: decoder {decoder_ser:.3f}, CTC {ctc_ser:.3f}. With a symbol missing, CTC wrote "
      f"what was heard in {summary['CTC']['wrote what was heard']}/40 cases, the decoder in "
      f"{summary['attention decoder']['wrote what was heard']}/40.")

### B · What a random split hides, and what augmentation buys

Now the data experiment. For every fold, train the same small classifier (log-mel features averaged over three time segments, a hidden layer of 32 units, Adam) three times:

1. **random clips:** folds made by shuffling clips, ignoring who spoke them;
2. **held-out speakers:** your `group_folds`, so every test speaker is new;
3. **held-out speakers + augmentation:** the same folds, with two augmented copies of every *training* clip (speed factor between 0.8 and 1.25, a shift of up to 50 ms, noise at 10 to 30 dB SNR).

In [ ]:
def clip_features(x):
    """Mean log-mel values in the first, middle and last third of the clip: 60 numbers."""
    return np.concatenate([part.mean(axis=0) for part in np.array_split(log_mel_spectrogram(x, SAMPLE_RATE), 3)])


def augment(train_clips, rng, copies=2):
    """The original clips plus `copies` augmented versions of each, built from your three functions."""
    out = [train_clips]
    for _ in range(copies):
        out.append(np.array([
            add_noise_at_snr(time_shift(fit_length(speed_change(x, rng.uniform(0.8, 1.25)), CLIP), int(rng.integers(-400, 401))),
                             rng.normal(size=CLIP), rng.uniform(10.0, 30.0))
            for x in train_clips]))
    return np.concatenate(out)


def train_and_score(train_x, train_y, test_x, test_y, seed=0, steps=300):
    """Standardise with training statistics, train a 60-32-4 network with Adam, return test accuracy."""
    mean, std = train_x.mean(axis=0), train_x.std(axis=0) + 1e-8
    a, b = (train_x - mean) / std, (test_x - mean) / std
    rng = np.random.default_rng(seed)
    W1, b1, W2, b2 = parameter(60, 32, rng=rng), zeros_parameter(32), parameter(32, 4, rng=rng, scale=0.1), zeros_parameter(4)
    opt, batches = Adam([W1, b1, W2, b2], learning_rate=0.01), np.random.default_rng(seed + 1)
    for _ in range(steps):
        rows = batches.choice(len(a), size=64, replace=False)
        loss = cross_entropy((Tensor(a[rows]) @ W1 + b1).relu() @ W2 + b2, train_y[rows])
        zero_grad([W1, b1, W2, b2])
        loss.backward()
        opt.step()
    predictions = np.argmax(np.maximum(0, b @ W1.data + b1.data) @ W2.data + b2.data, axis=1)
    return float(np.mean(predictions == test_y))


start = time.perf_counter()
all_features = np.array([clip_features(x) for x in clips])
n_clips = len(clips)
random_folds = np.array_split(np.random.default_rng(100).permutation(n_clips), 4)
scores = {"random clips": [], "held-out speakers": [], "held-out speakers + augmentation": []}
for f in range(4):
    test_index = random_folds[f]
    train_index = np.setdiff1d(np.arange(n_clips), test_index)
    scores["random clips"].append(train_and_score(all_features[train_index], words[train_index],
                                                  all_features[test_index], words[test_index]))
    test_index = folds[f]
    train_index = np.setdiff1d(np.arange(n_clips), test_index)
    assert not set(speaker_ids[train_index]) & set(speaker_ids[test_index]), "a speaker is on both sides of the split"
    scores["held-out speakers"].append(train_and_score(all_features[train_index], words[train_index],
                                                       all_features[test_index], words[test_index]))
    augmented = augment(clips[train_index], np.random.default_rng(7 + f))
    augmented_features = np.array([clip_features(x) for x in augmented])
    scores["held-out speakers + augmentation"].append(train_and_score(augmented_features, np.tile(words[train_index], 3),
                                                                      all_features[test_index], words[test_index]))
means = {name: float(np.mean(values)) for name, values in scores.items()}
for name, values in scores.items():
    print(f"{name:33s}: mean accuracy {means[name]:.3f}   folds {np.round(values, 3)}")
print(f"(4 folds of 80 test clips; {time.perf_counter() - start:.1f} s)")

In [ ]:
gap = means["random clips"] - means["held-out speakers"]
gain = means["held-out speakers + augmentation"] - means["held-out speakers"]
assert gap >= 0.05, "a random clip split should overstate accuracy on new speakers"
assert gain >= 0.05, "speed, shift and noise augmentation should help on new speakers"
print(f"✅ A random split overstated new-speaker accuracy by {gap:.2f}; augmentation raised new-speaker accuracy by {gain:.2f}.")

**What to notice.** The random split reports the highest accuracy, but most of its test clips come from speakers whose exact pitch range the model has already seen. On new speakers the same model does clearly worse, because a new base pitch moves every contour into mel bands it has seen less often. Speed perturbation moves the training contours up and down in pitch, which is precisely the variation between these speakers, and here the gap closes completely. That is partly by design: these speakers differ mainly in pitch and rate, which speed perturbation imitates well. With real speech the direction of both effects is typical, but their sizes depend on the data: measure them, do not assume them.

**Honest scope.** Sixteen generated "speakers" and four contour words are a model of the problem, not a speech corpus. Real systems adapt to new speakers and conditions with thousands of speakers in training, speed perturbation and noise from recorded environments, SpecAugment-style masking of features, and sometimes fine-tuning on a little data from the target domain, which must come from people who have agreed to that use.

## Exercises

### Exercise 1: `speaker_split`

Write `speaker_split(groups, validation_groups, test_groups)` that returns a dictionary with keys `"train"`, `"validation"` and `"test"`, each holding the sorted clip indices for that partition. Clips whose group is in `test_groups` go to test, those in `validation_groups` to validation, and all the others to training. This is the manifest you would save next to a dataset.

In [ ]:
def speaker_split_reference(groups, validation_groups, test_groups):
    test = np.isin(groups, test_groups)
    validation = np.isin(groups, validation_groups) & ~test
    return {"train": np.flatnonzero(~test & ~validation), "validation": np.flatnonzero(validation), "test": np.flatnonzero(test)}

In [ ]:
def speaker_split(groups, validation_groups, test_groups):
    """{"train": ..., "validation": ..., "test": ...} sorted index arrays, with whole groups in each."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
speaker_split = practice("speaker_split", speaker_split, speaker_split_reference, cases=[
    (np.array([0, 1, 2, 0, 1, 2, 3]), [1], [2]),
    (speaker_ids, [0, 1], [2, 3, 4]),
])

### Exercise 2: `random_gain`

Recording levels differ, so another cheap augmentation scales the waveform by a gain in decibels: $x \cdot 10^{g/20}$. Write `random_gain(signal, gain_db)` (the caller chooses `gain_db` at random). Would this augmentation change anything for a model that uses standardised **log**-mel features? Think about what a constant factor becomes after a logarithm.

In [ ]:
def random_gain_reference(signal, gain_db):
    return signal * 10 ** (gain_db / 20)

In [ ]:
def random_gain(signal, gain_db):
    """Scale the waveform by gain_db decibels."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
random_gain = practice("random_gain", random_gain, random_gain_reference, cases=[
    (np.array([1.0, -0.5]), 20.0), (np.array([1.0, -0.5]), -6.0), (np.zeros(3), 10.0),
])

### Exercise 3: exposure bias, by hand

Copy `generate` into a new cell and change it so that the first generated token is replaced by a **wrong** symbol before decoding continues. On the clean test set, how many of the remaining symbols does the decoder still get right? What does that tell you about training only with teacher forcing? (Scheduled sampling, which sometimes feeds the model its own predictions during training, is one remedy.)

### Exercise 4: an open question

The augmented model was trained on speed factors from 0.8 to 1.25. Suppose a deployment adds speakers whose base pitch is 150 Hz, below every training speaker. Which augmentation setting would you change, how would you check that the change helps without touching the final test speakers, and what would you report if it did not?

## Common errors and debugging

- **The decoder's loss is near zero, but generation is poor.** Teacher forcing hides exposure bias; always evaluate with `generate`, never with teacher-forced accuracy.
- **Generation never stops or stops at once.** `</s>` is missing from the training targets, or padding steps are not masked, so the model learns to predict padding.
- **Attention weights that do not sum to one, or `nan`.** The softmax is over the wrong axis (it must be over the frames), or the largest score was not subtracted.
- **The measured SNR is off by a factor of ten in decibels.** Power ratios use $10 \log_{10}$; amplitude ratios use $20 \log_{10}$.
- **A speed change that alters pitch but not duration.** The resampled clip was cropped or padded back to the original length before measuring; that is fine for a fixed-size model input, but report it.
- **Test accuracy that drops when you switch to speaker folds.** That is not a bug: the random split was measuring something easier.
- **Augmented clips in the test fold.** Augment after splitting, and only the training side.

## Summary

An encoder–decoder recogniser predicts one token at a time, attends to the frames it needs, and decides when to stop; trained with teacher forcing, it learns both the acoustics and the regularities of the transcripts. That prior helps on noisy audio and misleads on audio that breaks it, where CTC's frame-local decisions simply report what was heard. For data, a split must match the claim: keep every speaker on one side to measure new-speaker accuracy, and expect it to be lower than a random split suggests. Augmentations that keep the labels true, such as noise at a set SNR, time shifts and speed perturbation by resampling, recover much of that gap.

## Next

**04 · Synthesis, audio tokens, streaming and evaluation**: run the pipeline in the other direction with a small vocoder, compress audio into tokens with a k-means codebook, process it in streaming chunks with a latency budget, and score recognition with word and character error rates, ending with the capstone scaffold.